# 4 · Outputs for the web

The same styled map can leave roadstyle in several shapes. The heart of it is the **canonical JSON spec** (`spec/1`): your data plus the *baked-in* per-edge style, ready for any frontend (Leaflet, MapLibre, deck.gl) to draw.

In [1]:
import pathlib
import geopandas as gpd
import roadstyle as rs

# A GeoDataFrame of road edges. roadstyle only needs a *road-class* column (default "highway")
# and line geometry in any CRS — it reprojects to web coordinates (EPSG:4326) for you.
edges = gpd.read_file(pathlib.Path("data") / "sundbyberg_edges.gpkg")
print(f"{len(edges):,} edges, CRS {edges.crs}")
edges[[c for c in edges.columns if c != edges.geometry.name]].head(3)


4,039 edges, CRS EPSG:4326


,highway,name,oneway,maxspeed
0,tertiary,NaN,NaN,40
1,tertiary,NaN,NaN,40
2,secondary,Enköpingsvägen,yes,50


In [2]:
# Maps/files in this guide are written to ./output/ (git-ignored), so nothing clutters the repo.
out = pathlib.Path("output"); out.mkdir(exist_ok=True)


### `to_spec` — the canonical JSON
Each feature carries reserved `__rs_*` properties (`__rs_fill`, `__rs_w`, `__rs_casing`, …) so the browser never needs roadstyle's styling logic.

In [3]:
spec = rs.to_spec(edges, theme="dark", color_by="highway")
print("top-level keys:", list(spec))
print("first feature props:", list(spec["geojson"]["features"][0]["properties"])[:10])

top-level keys: ['roadstyle', 'crs', 'theme', 'bounds', 'render', 'basemap', 'tooltip', 'legend', 'geojson']
first feature props: ['highway', 'name', 'oneway', 'maxspeed', '__rs_fill', '__rs_w', '__rs_op', '__rs_dash', '__rs_casing', '__rs_cw']


### `save_spec` / `load_spec` — round-trip to a file

In [4]:
rs.save_spec(spec, out / "map_data.json")
again = rs.load_spec(out / "map_data.json")
print("reloaded", len(again["geojson"]["features"]), "features")

reloaded 4039 features


### `to_geojson` — just the styled features

In [5]:
gj = rs.to_geojson(edges, color_by="length_m" if "length_m" in edges else "highway")
print(gj["type"], "with", len(gj["features"]), "features")

FeatureCollection with 4039 features


### `to_html` / `to_iframe` / `save` — ready-to-view HTML
`full=False` gives a `<div>+<script>` fragment to inject into an existing page; `to_iframe` wraps a full page in a sandboxed iframe; `save` writes a standalone file.

In [6]:
from IPython.display import HTML
rs.save(edges, out / "standalone.html", theme="dark")          # double-click to open
HTML(rs.to_iframe(edges, theme="dark", height="360px"))         # preview inline

/home/kaveh/miniconda3/envs/roadstyle/lib/python3.11/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


For a non-folium site, hand the JSON spec to the reusable JS renderer — see notebook 5.